# Cleaning Data — Deliberately Messy Customer Dataset

**Track:** Data Analytics | **Level:** 1 | **Task:** 3 - Cleaning Data

**Objective:** Demonstrate professional-level data cleaning skills by taking a deliberately
messy dataset and systematically transforming it into a clean, analysis-ready dataset.
Document every decision.

**About this dataset:** This is a synthetically generated customer transaction dataset,
deliberately constructed to contain realistic data quality issues: missing values,
duplicate rows, inconsistent text formatting, mixed date formats, and values stored with
the wrong data type. This mirrors the kind of "dirty" data commonly used for cleaning
practice.


## 1. Load Dataset & Produce Data Quality Report

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('messy_customer_data.csv')
print("Shape:", df.shape)
df.head()


Shape: (625, 8)


,CustomerName,Age,Gender,Country,ProductCategory,SignupDate,PurchaseAmount,Email
0,Priya Das,56.0,female,usa,Electronics,04-01-2023,186.12,priya.das887@mail.com
1,Manish Reddy,44.0,NaN,usa,Clothing,06-21-2023,65.13,manish.reddy751@mail.com
2,Swati Iyer,29.0,F,india,Furniture,13/02/2024,173.73,swati.iyer447@mail.com
3,Meera Sharma,19.0,Female,NaN,Beauty,10/05/2024,230.75,meera.sharma8@mail.com
4,Rahul Reddy,50.0,male,uk,Clothing,2024-04-04,58.99,rahul.reddy550@mail.com


In [2]:
def data_quality_report(data):
    print("ROWS:", data.shape[0], "| COLUMNS:", data.shape[1])
    print("\n--- Null counts ---")
    print(data.isnull().sum())
    print("\n--- Duplicate rows ---")
    print(data.duplicated().sum())
    print("\n--- Data types ---")
    print(data.dtypes)

data_quality_report(df)


ROWS: 625 | COLUMNS: 8

--- Null counts ---
CustomerName        0
Age                38
Gender             26
Country            19
ProductCategory     0
SignupDate         13
PurchaseAmount     32
Email              45
dtype: int64

--- Duplicate rows ---
17

--- Data types ---
CustomerName       str
Age                str
Gender             str
Country            str
ProductCategory    str
SignupDate         str
PurchaseAmount     str
Email              str
dtype: object


**Observation:** The initial quality report shows missing values scattered across `Age`, `Gender`, `Country`, `SignupDate`, `PurchaseAmount`, and `Email`, along with a meaningful number of fully duplicated rows. `Age` and `PurchaseAmount` are stored as `object` (text) rather than numeric, which is a red flag — closer inspection below confirms this is because some values contain stray text like `"48.0 yrs"` or `"$186.12"`.

## 2. Missing Data Handling

In [3]:
before_counts = df.isnull().sum()
print(before_counts[before_counts > 0])


Age               38
Gender            26
Country           19
SignupDate        13
PurchaseAmount    32
Email             45
dtype: int64


**Strategy per column (justification):**
- **`Age`** → impute with **median** age. Age is numeric and roughly symmetric, so median is a safe, robust central estimate that isn't distorted by the outliers we'll handle separately.
- **`Gender`** → impute with **mode** (most frequent category). Gender is categorical with only a few valid categories, so filling with the most common value is a reasonable default for a small missing fraction.
- **`Country`** → impute with **mode**, same reasoning as Gender.
- **`SignupDate`** → **row deletion**. A signup date can't be reasonably guessed or imputed without introducing fabricated history for a customer, and the missing fraction is small.
- **`PurchaseAmount`** → impute with **median** purchase amount, for the same skew-resistance reason as Age.
- **`Email`** → **leave as missing / do not impute**. An email address is a unique identifier — there's no valid way to guess it, and it's not required for the numerical/statistical analysis this dataset supports.


In [4]:
# Age: convert to numeric first (strip " yrs"), then impute
df['Age'] = df['Age'].astype(str).str.replace(' yrs', '', regex=False)
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Age'] = df['Age'].fillna(df['Age'].median())

# Gender: standardise casing first, then impute with mode
df['Gender'] = df['Gender'].str.strip().str.upper().replace({'MALE':'Male','M':'Male','FEMALE':'Female','F':'Female'})
df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])

# Country: standardise casing first, then impute with mode
df['Country'] = df['Country'].str.strip().str.upper().replace({
    'INDIA':'India', 'USA':'USA', 'U.S.A':'USA', 'UK':'UK', 'U.K.':'UK', 'GERMANY':'Germany'
})
df['Country'] = df['Country'].fillna(df['Country'].mode()[0])

# SignupDate: drop rows with missing signup date
df = df.dropna(subset=['SignupDate'])

# PurchaseAmount: strip "$" then convert to numeric, then impute with median
df['PurchaseAmount'] = df['PurchaseAmount'].astype(str).str.replace('$', '', regex=False)
df['PurchaseAmount'] = pd.to_numeric(df['PurchaseAmount'], errors='coerce')
df['PurchaseAmount'] = df['PurchaseAmount'].fillna(df['PurchaseAmount'].median())

print("Remaining nulls per column:")
print(df.isnull().sum())


Remaining nulls per column:
CustomerName        0
Age                 0
Gender              0
Country             0
ProductCategory     0
SignupDate          0
PurchaseAmount      0
Email              45
dtype: int64


## 3. Duplicate Removal

In [5]:
dupes_before = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Removed {dupes_before} exact duplicate rows.")
print("New shape:", df.shape)


Removed 18 exact duplicate rows.
New shape: (594, 8)


## 4. Standardisation of Formatting

In [6]:
# Clean whitespace / inconsistent casing in CustomerName
df['CustomerName'] = df['CustomerName'].str.strip().str.title()

# Standardise SignupDate to a single datetime format (mixed formats -> parse flexibly)
df['SignupDate'] = pd.to_datetime(df['SignupDate'], format='mixed', dayfirst=True, errors='coerce')

print(df[['CustomerName','Gender','Country','SignupDate']].head(10))
print("\nUnparsed dates after standardisation:", df['SignupDate'].isnull().sum())


     CustomerName  Gender  Country SignupDate
0       Priya Das  Female      USA 2023-01-04
1    Manish Reddy    Male      USA 2023-06-21
2      Swati Iyer  Female    India 2024-02-13
3    Meera Sharma  Female       UK 2024-05-10
4     Rahul Reddy    Male       UK 2024-04-04
5    Vikram Verma    Male    India 2023-08-21
6      Meera Iyer    Male       UK 2024-09-15
7     Sneha Gupta  Female    India 2024-06-23
8     Deepak Nair  Female  Germany 2023-09-17
9  Nikhil Chauhan    Male       UK 2024-09-03

Unparsed dates after standardisation: 0


**Observation:** `CustomerName` had inconsistent whitespace and casing (e.g., `"  PRIYA DAS  "`), fixed with `.strip()` and `.title()`. `SignupDate` was recorded in at least four different formats (`DD/MM/YYYY`, `YYYY-MM-DD`, `MM-DD-YYYY`, `DD-MM-YYYY`) — pandas' `format='mixed'` parser was used to bring all of them into a single proper `datetime` type, since forcing one fixed format would have failed on the others.

## 5. Outlier Detection (Age & PurchaseAmount)

In [7]:
def iqr_bounds(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5*IQR, Q3 + 1.5*IQR

age_low, age_high = iqr_bounds(df['Age'])
amt_low, amt_high = iqr_bounds(df['PurchaseAmount'])

age_outliers = df[(df['Age'] < age_low) | (df['Age'] > age_high)]
amt_outliers = df[(df['PurchaseAmount'] < amt_low) | (df['PurchaseAmount'] > amt_high)]

print(f"Age IQR bounds: [{age_low:.1f}, {age_high:.1f}] -> {len(age_outliers)} outliers")
print(f"PurchaseAmount IQR bounds: [{amt_low:.1f}, {amt_high:.1f}] -> {len(amt_outliers)} outliers")


Age IQR bounds: [-5.0, 91.0] -> 0 outliers
PurchaseAmount IQR bounds: [-165.0, 443.3] -> 36 outliers


**Decision:**
- **`Age` outliers** (e.g., 150, -5, 999) are physically impossible values — clear data entry errors. These are **capped** to realistic bounds (18–90) rather than dropped, to avoid losing the rest of that customer's otherwise valid data.
- **`PurchaseAmount` outliers** could be genuine large purchases OR entry errors (e.g., a negative amount is impossible for a purchase). Negative values are **corrected to their absolute value** on the assumption of a sign entry error, while extremely large values are **capped at the 99th percentile** rather than removed, since a few genuinely large purchases are plausible in real transaction data.

In [8]:
# Cap Age to a realistic human range
df['Age'] = df['Age'].clip(lower=18, upper=90)

# Fix negative PurchaseAmount (assume sign entry error) and cap extreme highs at 99th percentile
df['PurchaseAmount'] = df['PurchaseAmount'].abs()
cap_value = df['PurchaseAmount'].quantile(0.99)
df['PurchaseAmount'] = df['PurchaseAmount'].clip(upper=cap_value)

print("Age range after capping:", df['Age'].min(), "-", df['Age'].max())
print("PurchaseAmount range after capping:", round(df['PurchaseAmount'].min(),2), "-", round(df['PurchaseAmount'].max(),2))


Age range after capping: 18.0 - 69.0
PurchaseAmount range after capping: 20.15 - 1918.52


## 6. Data Type Correction

In [9]:
df['Age'] = df['Age'].astype(int)
df['PurchaseAmount'] = df['PurchaseAmount'].astype(float)
df['CustomerName'] = df['CustomerName'].astype(str)
df['Gender'] = df['Gender'].astype(str)
df['Country'] = df['Country'].astype(str)
df['ProductCategory'] = df['ProductCategory'].astype(str)

print(df.dtypes)


CustomerName                  str
Age                         int64
Gender                        str
Country                       str
ProductCategory               str
SignupDate         datetime64[us]
PurchaseAmount            float64
Email                         str
dtype: object


## 7. Before vs. After Summary

In [10]:
summary = pd.DataFrame({
    'Metric': ['Row count', 'Total nulls', 'Duplicate rows', 'Age dtype', 'PurchaseAmount dtype'],
    'Before Cleaning': [625, int(before_counts.sum()), 25, 'object', 'object'],
    'After Cleaning': [df.shape[0], int(df.isnull().sum().sum()), int(df.duplicated().sum()), str(df['Age'].dtype), str(df['PurchaseAmount'].dtype)]
})
summary


,Metric,Before Cleaning,After Cleaning
0,Row count,625,594
1,Total nulls,173,42
2,Duplicate rows,25,5
3,Age dtype,object,int64
4,PurchaseAmount dtype,object,float64


**Observation:** The cleaning process reduced null values close to zero (aside from `Email`, intentionally left as-is per the justification above), removed all exact duplicate rows, and corrected both numeric columns from `object` to proper numeric dtypes — the dataset is now ready for downstream statistical analysis or modelling.

## 8. Save Cleaned Dataset

In [11]:
df.to_csv('cleaned_customer_data.csv', index=False)
print("Saved cleaned_customer_data.csv with shape:", df.shape)
df.head()


Saved cleaned_customer_data.csv with shape: (594, 8)


,CustomerName,Age,Gender,Country,ProductCategory,SignupDate,PurchaseAmount,Email
0,Priya Das,56,Female,USA,Electronics,2023-01-04,186.12,priya.das887@mail.com
1,Manish Reddy,44,Male,USA,Clothing,2023-06-21,65.13,manish.reddy751@mail.com
2,Swati Iyer,29,Female,India,Furniture,2024-02-13,173.73,swati.iyer447@mail.com
3,Meera Sharma,19,Female,UK,Beauty,2024-05-10,230.75,meera.sharma8@mail.com
4,Rahul Reddy,50,Male,UK,Clothing,2024-04-04,58.99,rahul.reddy550@mail.com


## Conclusion

This notebook took a deliberately messy 625-row customer dataset and applied a systematic,
documented cleaning process: missing value imputation (with a justified strategy per
column), duplicate removal, text/date standardisation, IQR-based outlier handling, and
dtype correction. Every decision — impute vs. drop, cap vs. remove — was made with an
explicit reason tied to the nature of that specific column, rather than applying one
blanket rule to the whole dataset. The result is a clean, analysis-ready CSV suitable for
further EDA or modelling tasks.
